### JSON Parsing and Processing

In [1]:
import json
import os
os.makedirs("data/json_files", exist_ok=True)

In [4]:
# Sample nested JSON data
json_data = {
    "company": "AK Unity",
    "employees": [
        {
            "id": 1,
            "name": "Aditya Kumar Singh",
            "role": "AI Engineer",
            "skills": ["Python", "GenerativeAI", "Machine Learning"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"},
            ],
        },
        {
            "id": 2,
            "name": "Connectwithaadi",
            "role": "RAG Engineer",
            "skills": ["AgenticAI", "GenerativeAI", "Machine Learning"],
            "projects": [
                {"name": "Multi AI Agent", "status": "In Progress"},
                {"name": "LLM Gateway", "status": "Completed"},
            ]
        }
    ],
    "department": {
        "engineering": {
            "head": "Mike Johnson",
            "budget": 100000,
            "team_size": 24
        },
        "data_scientist": {
            "head": "sania khan",
            "budget": 456666,
            "team_size": 15
        }
    }
}

In [5]:
json_data

{'company': 'AK Unity',
 'employees': [{'id': 1,
   'name': 'Aditya Kumar Singh',
   'role': 'AI Engineer',
   'skills': ['Python', 'GenerativeAI', 'Machine Learning'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Connectwithaadi',
   'role': 'RAG Engineer',
   'skills': ['AgenticAI', 'GenerativeAI', 'Machine Learning'],
   'projects': [{'name': 'Multi AI Agent', 'status': 'In Progress'},
    {'name': 'LLM Gateway', 'status': 'Completed'}]}],
 'department': {'engineering': {'head': 'Mike Johnson',
   'budget': 100000,
   'team_size': 24},
  'data_scientist': {'head': 'sania khan', 'budget': 456666, 'team_size': 15}}}

In [6]:
with open('data/json_files/company_data.json','w') as f:
    json.dump(json_data, f, indent=2)

In [7]:
# Save Json Line format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 111},
    {"timestamp": "2025-01-01", "event": "page_view", "user_id": 222, "page": "/home"},
    {"timestamp": "2026-01-01", "event": "purchase", "user_id": 333, "amount": 99.43}
]

with open('data/json_files/events.jsonl', 'w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

### Json Processing Strategies

In [9]:

!uv pip install jq

Using Python 3.13.15 environment at: D:\Mission Project\RAG\.venv
Resolved 1 package in 274ms
Prepared 1 package in 130ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 13ms
 + jq==1.12.0


In [11]:
from langchain_community.document_loaders import JSONLoader
import json

## Method 1: JSONLoader with jq_schema
print(" 1 JSONLoader - Extract specific fields")

# Extract employee information
employee_loader = JSONLoader(
    file_path='data/json_files/company_data.json',
    jq_schema='.employees[]',   # jq query to extract each employee
    text_content=False  # Get full JSON objects
)

employee_docs = employee_loader.load()
print(f"Loaded {len(employee_docs)} employee documents")
print(f"First employee: {employee_docs[0].page_content[:200]}....")
print(employee_docs)

 1 JSONLoader - Extract specific fields
Loaded 2 employee documents
First employee: {"id": 1, "name": "Aditya Kumar Singh", "role": "AI Engineer", "skills": ["Python", "GenerativeAI", "Machine Learning"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pi....
[Document(metadata={'source': 'D:\\Mission Project\\RAG\\0-DataIngestParsing\\data\\json_files\\company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "Aditya Kumar Singh", "role": "AI Engineer", "skills": ["Python", "GenerativeAI", "Machine Learning"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': 'D:\\Mission Project\\RAG\\0-DataIngestParsing\\data\\json_files\\company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Connectwithaadi", "role": "RAG Engineer", "skills": ["AgenticAI", "GenerativeAI", "Machine Learning"], "projects": [{"name": "Multi AI Agent", "status": "In Progress"

In [15]:
# Method 2: Custom JSON processing for complex structures
from typing import List
from langchain_core.documents import Document
print("\n 2 Custom JSON Processing")

def process_json_intelligently(filepath: str) -> List[Document]:
    """Process JSON with intelligent flattening and context preservation"""
    with open(filepath, 'r') as f:
        data = json.load(f)

    documents = []

    # Strategy 1: Create documents for each employee with full context
    for emp in data.get('employees',[]):
        content = f"""Employee Profile:
        Name: {emp['name']}
        Role: {emp['role']}
        Skills: {', '.join(emp['skills'])}
        
        Projects:"""
        for proj in emp.get('projects', []):
            content += f"\n- {proj['name']} (Status: {proj['status']})"

        doc = Document(
            page_content = content,
            metadata={
                'source': filepath,
                'data_type': 'employee_profile',
                'employee_id': emp['id'],
                'employee_name': emp['name'],
                'role': emp['role']
            }
        )
        documents.append(doc)


 2 Custom JSON Processing


In [16]:
process_json_intelligently("data/json_files/company_data.json")